# Gradient Descent to ODE

Chapter 20: ODE, SDE, and Continuous Limits of Algorithms · *Mathematics for AI and Machine Learning* by Fuheng Wu

Compare discrete gradient descent iterates with the continuous gradient-flow trajectory.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-20/gd-to-ode/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch

# The demo's defaults: f(x) = a x^2 / 2, step eta, start x0, number of steps
a, eta, x0, K = 1.2, 0.4, 2.0, 60

# ── Gradient descent is Euler's method on the gradient flow dx/dt = -f'(x) ──
#    x_{k+1} = x_k - eta a x_k  vs  x(t) = x0 e^{-a t} at t = k eta
print("  k     t      GD x_k          flow x(k eta)    ratio")
for k in (0, 1, 2, 5, 10, 20, 40, K):
    xd = x0 * (1 - eta * a) ** k
    xc = x0 * math.exp(-a * k * eta)
    print(f"{k:3d}  {k * eta:5.1f}   {xd:.6e}    {xc:.6e}     {xd / xc if xc else float('nan'):.4f}")

# ── Smaller steps track the flow better: error at a fixed time t = 2 ────────
print("\n  eta     GD at t=2      flow at t=2    |error|")
for e in (0.4, 0.2, 0.1, 0.05, 0.01):
    n = round(2 / e)
    xd = x0 * (1 - e * a) ** n
    print(f"{e:6.2f}   {xd:.6f}       {x0 * math.exp(-2 * a):.6f}       {abs(xd - x0 * math.exp(-2 * a)):.2e}")
# The error halves with eta: Euler is first order.

# ── Same picture with autograd on a 2D quadratic ────────────────────────────
H = torch.tensor([[1.2, 0.0], [0.0, 3.0]], dtype=torch.float64)
x = torch.tensor([2.0, 1.0], dtype=torch.float64, requires_grad=True)
opt = torch.optim.SGD([x], lr=0.1)
for _ in range(20):
    opt.zero_grad()
    (0.5 * x @ H @ x).backward()
    opt.step()
flow = torch.linalg.matrix_exp(-H * 2.0) @ torch.tensor([2.0, 1.0], dtype=torch.float64)
print("\n2D: GD after 20 steps of 0.1 =", [round(v, 5) for v in x.tolist()], " flow at t=2 =", [round(v, 5) for v in flow.tolist()])

  k     t      GD x_k          flow x(k eta)    ratio
  0    0.0   2.000000e+00    2.000000e+00     1.0000
  1    0.4   1.040000e+00    1.237567e+00     0.8404
  2    0.8   5.408000e-01    7.657858e-01     0.7062
  5    2.0   7.604081e-02    1.814359e-01     0.4191
 10    4.0   2.891102e-03    1.645949e-02     0.1756
 20    8.0   4.179236e-06    1.354575e-04     0.0309
 40   16.0   8.733006e-12    9.174363e-09     0.0010
 60   24.0   1.824864e-17    6.213680e-13     0.0000

  eta     GD at t=2      flow at t=2    |error|
  0.40   0.076041       0.181436       1.05e-01
  0.20   0.128578       0.181436       5.29e-02
  0.10   0.155126       0.181436       2.63e-02
  0.05   0.168323       0.181436       1.31e-02
  0.01   0.178821       0.181436       2.61e-03



2D: GD after 20 steps of 0.1 = [0.15513, 0.0008]  flow at t=2 = [0.18144, 0.00248]
